# FINA 4460 Group Project
## Mean-Reverting Strategy with Technical Indicators

### Indicators:
1. Mean Reversion (Johnson-SU percentile thresholds) - from class
2. RSI (Relative Strength Index)
3. ATR (Average True Range)
4. [Choice Indicator - TBD]

---

## Cell 0: Imports

In [1]:
import requests
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import johnsonsu
from scipy.optimize import minimize
import time
import csv
from dotenv import load_dotenv
import os

load_dotenv("secrets.env")

api_key = os.getenv("API_KEY")
db_password = os.getenv("DB_PASSWORD")
debug = os.getenv("DEBUG")

## Cell 1: Configuration

In [3]:
# API Configuration
API_KEY = api_key

# Data Parameters - 3 stocks + 1 index (per assignment requirement 5)
TICKERS = ["AAPL", "MSFT", "GOOGL"]  # 3 different stocks
INDEX = "SPY"                         # S&P 500 index
ALL_SYMBOLS = TICKERS + [INDEX]       # Combined list for iteration

START_DATE = "2022-01-01"  # Assignment requires 2022 onwards
END_DATE = "2026-01-30"

# Estimation vs Test Split
EST_START = "2022-01-01"
EST_END = "2024-12-31"
# Test period: 2025-01-01 onwards

# Indicator Parameters
RSI_PERIOD = 14
RSI_OVERSOLD = 30
RSI_OVERBOUGHT = 70

ATR_PERIOD = 14

MEAN_REV_THRESHOLD = 0.05  # 5th and 95th percentile

---
## PART 1: DATA FUNCTIONS (Reused from class)
---

## Cell 2: Download Daily Data

In [4]:
def get_daily_data(startd, endd, ticker, api_key):
    """
    Download daily adjusted close prices from AlphaVantage.
    Returns: list of [date, ticker, adjusted_close]
    """
    info = []
    ticker_param = '&symbol=' + str(ticker)
    endpoint = 'function=TIME_SERIES_DAILY_ADJUSTED'
    size = '&outputsize=full'
    web = 'https://www.alphavantage.co/query?'
    url = web + endpoint + ticker_param + size + '&apikey=' + api_key

    r = requests.get(url)
    time.sleep(1)  # Rate limiting

    data = r.json()
    r1 = data.get('Time Series (Daily)', {})
    
    for d, values in sorted(r1.items()):
        if startd <= d <= endd:
            info.append([
                d,
                ticker,
                float(values['5. adjusted close']),
                float(values['3. low']),
                float(values['2. high']),
                float(values['1. open'])
            ])
    
    print(f"Downloaded {len(info)} days of data for {ticker}")
    return info

## Cell 3: Compute Log Returns

In [80]:
def compute_log_returns(data):
    """
    Compute daily log returns.
    Input: list of [date, ticker, close, low, high, open]
    Output: list of [date, ticker, close, low, high, open, log_return]
    """
    sorted_data = sorted(data, key=lambda x: x[0])
    data_with_returns = []

    for i in range(len(sorted_data)):
        if i == 0:
            data_with_returns.append(sorted_data[i] + [None])
        else:
            log_return = np.log(sorted_data[i][2]) - np.log(sorted_data[i-1][2])
            data_with_returns.append(sorted_data[i] + [log_return])

    return data_with_returns

## Cell 4: Partition Data (Train/Test Split)

In [81]:
def partition_by_date(data, est_start, est_end):
    """
    Split data into estimation (training) and testing samples.
    """
    est = [row for row in data if est_start <= row[0] <= est_end]
    test = [row for row in data if row[0] > est_end]
    print(f"Estimation sample: {len(est)} days")
    print(f"Test sample: {len(test)} days")
    return est, test

---
## PART 2: INDICATOR FUNCTIONS
Each indicator returns a list of signals: -1 (short), 0 (hold), 1 (long)

---

## Cell 5: Mean Reversion Signal (from class)

In [82]:
def moment_objective(params, sample_moments):
    """Objective function for Johnson-SU moment matching."""
    gamma, delta, xi, lam = params
    if delta <= 0 or lam <= 0:
        return 1e12
    try:
        fitted = johnsonsu.stats(gamma, delta, loc=xi, scale=lam, moments='mvsk')
        error = np.sum((np.array(fitted) - np.array(sample_moments))**2)
        return error
    except:
        return 1e12


def fit_johnson_su(returns):
    """
    Fit Johnson-SU distribution to returns using MLE.
    Returns: gamma, delta, xi, lam
    """
    returns = np.array([r for r in returns if r is not None], dtype=float)
    gamma, delta, xi, lam = johnsonsu.fit(returns)
    return gamma, delta, xi, lam


def signal_mean_reversion(data, gamma, delta, xi, lam, threshold=0.05):
    """
    Generate mean reversion signals based on Johnson-SU percentiles.
    
    Signal logic:
      - return < lower_threshold (5th pct) -> expect reversion UP -> BUY (1)
      - return > upper_threshold (95th pct) -> expect reversion DOWN -> SELL (-1)
      - else -> HOLD (0)
    
    Input: data with [date, ticker, close, low, high, open, log_return]
    Output: list of signals [-1, 0, 1] for each row
    """
    lower = float(johnsonsu.ppf(threshold, gamma, delta, loc=xi, scale=lam))
    upper = float(johnsonsu.ppf(1 - threshold, gamma, delta, loc=xi, scale=lam))
    
    print(f"Mean Reversion thresholds: lower={lower:.6f}, upper={upper:.6f}")
    
    signals = []
    for row in data:
        ret = row[6]  # log_return
        if ret is None:
            signals.append(0)
        elif ret < lower:
            signals.append(1)   # Buy on extreme negative return (mean reversion)
        elif ret > upper:
            signals.append(-1)  # Sell on extreme positive return (mean reversion)
        else:
            signals.append(0)
    
    return signals

## Cell 6: RSI Signal

In [84]:
def compute_rsi(data, period=14):
    """
    Compute Relative Strength Index.
    Input: data with [date, ticker, close, low, high, open, log_return]
    Output: list of RSI values (None for first 'period' entries)
    """
    closes = [row[2] for row in data]
    rsi_values = [None] * len(closes)
    
    for i in range(period, len(closes)):
        gains = []
        losses = []
        
        for j in range(i - period + 1, i + 1):
            change = closes[j] - closes[j - 1]
            if change > 0:
                gains.append(change)
                losses.append(0)
            else:
                gains.append(0)
                losses.append(abs(change))
        
        avg_gain = np.mean(gains)
        avg_loss = np.mean(losses)
        
        if avg_loss == 0:
            rsi_values[i] = 100
        else:
            rs = avg_gain / avg_loss
            rsi_values[i] = 100 - (100 / (1 + rs))
    
    return rsi_values


def signal_rsi(data, period=14, oversold=30, overbought=70):
    """
    Generate RSI signals.
    
    Signal logic:
      - RSI < oversold (30) -> oversold -> BUY (1)
      - RSI > overbought (70) -> overbought -> SELL (-1)
      - else -> HOLD (0)
    
    Output: list of signals [-1, 0, 1]
    """
    rsi_values = compute_rsi(data, period)
    
    signals = []
    for rsi in rsi_values:
        if rsi is None:
            signals.append(0)
        elif rsi < oversold:
            signals.append(1)   # Buy when oversold
        elif rsi > overbought:
            signals.append(-1)  # Sell when overbought
        else:
            signals.append(0)
    
    return signals, rsi_values

## Cell 7: ATR Signal

In [85]:
def compute_atr(data, period=14):
    """
    Compute ATR using Wilder's method:
    - First ATR = average of first 'period' true ranges
    - Thereafter: ATR_t = ((ATR_{t-1} * (period - 1)) + TR_t) / period
    """
    atr_values = [None] * len(data)
    true_ranges = [None] * len(data)

    # Compute True Range starting at i=1
    for i in range(1, len(data)):
        high = data[i][4]
        low = data[i][3]
        prev_close = data[i - 1][2]

        tr = max(
            high - low,
            abs(high - prev_close),
            abs(low - prev_close)
        )
        true_ranges[i] = tr

    # Need at least 'period' TR values to initialize ATR
    if len(data) <= period:
        return atr_values

    # First ATR = simple average of first 'period' TR values
    first_tr_window = [tr for tr in true_ranges[1:period+1] if tr is not None]
    if len(first_tr_window) == period:
        atr_values[period] = np.mean(first_tr_window)

    # Wilder smoothing for subsequent ATR values
    for i in range(period + 1, len(data)):
        tr = true_ranges[i]
        prev_atr = atr_values[i - 1]
        if tr is not None and prev_atr is not None:
            atr_values[i] = ((prev_atr * (period - 1)) + tr) / period

    return atr_values

def signal_atr(data, period=14, multiplier=1.5):
    """
    Generate ATR breakout signals.
    
    Signal logic:
      - Price > prev_close + (ATR * multiplier) -> breakout UP -> BUY (1)
      - Price < prev_close - (ATR * multiplier) -> breakout DOWN -> SELL (-1)
      - else -> HOLD (0)
    
    Output: list of signals [-1, 0, 1]
    """
    atr_values = compute_atr(data, period)
    
    signals = []
    for i in range(len(data)):
        if i == 0 or atr_values[i] is None:
            signals.append(0)
        else:
            close = data[i][2]
            prev_close = data[i-1][2]
            atr = atr_values[i]
            
            upper_band = prev_close + (atr * multiplier)
            lower_band = prev_close - (atr * multiplier)
            
            if close > upper_band:
                signals.append(1)   # Breakout up
            elif close < lower_band:
                signals.append(-1)  # Breakout down
            else:
                signals.append(0)
    
    return signals, atr_values

## Cell 8: Choice Indicator (Placeholder - Bollinger Bands)

In [86]:
def signal_bollinger_bands(data, period=20, num_std=2):
    """
    Generate Bollinger Bands signals.
    
    Signal logic (mean reversion):
      - Price < lower band -> BUY (1)
      - Price > upper band -> SELL (-1)
      - else -> HOLD (0)
    
    Output: list of signals [-1, 0, 1]
    """
    closes = [row[2] for row in data]
    signals = [0] * len(data)
    
    for i in range(period, len(data)):
        window = closes[i-period:i]
        ma = np.mean(window)
        std = np.std(window)
        
        upper_band = ma + (num_std * std)
        lower_band = ma - (num_std * std)
        
        if closes[i] < lower_band:
            signals[i] = 1   # Buy below lower band
        elif closes[i] > upper_band:
            signals[i] = -1  # Sell above upper band
        else:
            signals[i] = 0
    
    return signals

---
## PART 3: COMBINE SIGNALS & EXECUTE STRATEGY
---

## Cell 9: Combine Signals into Recommendation

In [87]:
def combine_signals(signal_list, method='core_confirm'):
    """
    Combine multiple indicator signals into a single recommendation.
    
    Input: list of signal lists in order [sig_mr, sig_rsi, sig_atr, sig_bb]
    
    Methods:
      - 'core_confirm': MR is core, needs RSI or BB confirmation (recommended)
      - 'weighted': weighted score with threshold
      - 'consensus': 2 of 3 mean-rev indicators must agree (excludes ATR)
      - 'majority': sign of sum (original)
    
    Output: list of combined signals [-1, 0, 1]
    """
    sig_mr, sig_rsi, sig_atr, sig_bb = signal_list
    n = len(sig_mr)
    combined = []
    
    for i in range(n):
        mr = sig_mr[i]
        rsi = sig_rsi[i]
        atr = sig_atr[i]
        bb = sig_bb[i]
        
        if method == 'core_confirm':
            # Rule 1: Mean Reversion is core, RSI or BB must confirm
            if mr == 1 and (rsi == 1 or bb == 1):
                combined.append(1)
            elif mr == -1 and (rsi == -1 or bb == -1):
                combined.append(-1)
            else:
                combined.append(0)
        
        elif method == 'weighted':
            # Rule 2: Weighted score (MR=2, RSI=1.5, BB=1, ATR=0)
            score = 2*mr + 1.5*rsi + 1*bb
            if score >= 2.5:
                combined.append(1)
            elif score <= -2.5:
                combined.append(-1)
            else:
                combined.append(0)
        
        elif method == 'consensus':
            # Rule 3: 2 of 3 mean-reversion indicators must agree
            buy_count = (mr == 1) + (rsi == 1) + (bb == 1)
            sell_count = (mr == -1) + (rsi == -1) + (bb == -1)
            if buy_count >= 2:
                combined.append(1)
            elif sell_count >= 2:
                combined.append(-1)
            else:
                combined.append(0)
        
        elif method == 'majority':
            # Original majority voting (all 4 indicators)
            total = mr + rsi + atr + bb
            if total > 0:
                combined.append(1)
            elif total < 0:
                combined.append(-1)
            else:
                combined.append(0)
        
        else:
            combined.append(0)
    
    return combined

## Cell 10: Compute Strategy Returns

In [88]:
def compute_strategy_returns(data, signals):
    """
    Compute strategy returns based on signals.
    Signal at t -> trade executed, return realized at t+1.
    
    Per assignment: "Only store the returns where there is a trade"
    When no position, strategy_return = None (not 0)
    
    Output: list of [date, ticker, close, log_return, signal, strategy_return]
    """
    results = []
    
    for i in range(len(data)):
        row = data[i]
        date = row[0]
        ticker = row[1]
        close = row[2]
        log_ret = row[6]
        signal = signals[i]
        
        if i == 0:
            strat_ret = None
        else:
            prev_signal = signals[i-1]
            if log_ret is None:
                strat_ret = None
            elif prev_signal == 1:
                strat_ret = log_ret      # Long position
            elif prev_signal == -1:
                strat_ret = -log_ret     # Short position
            else:
                strat_ret = None         # No position - per assignment, don't store
        
        results.append([date, ticker, close, log_ret, signal, strat_ret])
    
    return results

## Cell 11: Export to CSV

In [89]:
def export_results(results, filename='strategy_results.csv'):
    """
    Export strategy results to CSV.
    """
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['date', 'ticker', 'close', 'log_return', 'signal', 'strategy_return'])
        writer.writerows(results)
    print(f"Saved results to {filename}")

---
## PART 4: PERFORMANCE METRICS
---

## Cell 12: Performance Metrics

In [90]:
def compute_performance_metrics(results, rf_annual=0.05):
    """
    Compute strategy performance metrics.
    
    Metrics:
      - Trades per month
      - Average return and t-test significance
      - Long/Short average returns
      - Cumulative return
      - Sharpe ratio
      - Sortino ratio
      - VaR at 5%
    """
    # Extract strategy returns (skip None)
    strat_rets = [row[5] for row in results if row[5] is not None]
    signals = [row[4] for row in results]
    dates = [row[0] for row in results]
    
    if len(strat_rets) == 0:
        print("No strategy returns to analyze.")
        return None
    
    strat_rets = np.array(strat_rets)
    
    # Daily risk-free rate
    rf_daily = rf_annual / 252
    
    # --- Basic Stats ---
    mean_ret = np.mean(strat_rets)
    std_ret = np.std(strat_rets, ddof=1)
    
    # --- Trades per month ---
    num_trades = sum(1 for s in signals if s != 0)
    months = len(set([d[:7] for d in dates]))  # unique YYYY-MM
    trades_per_month = num_trades / max(months, 1)
    
    # --- T-test for significance ---
    t_stat, p_value = stats.ttest_1samp(strat_rets, 0)
    
    # --- Long/Short average returns ---
    long_rets = [results[i][5] for i in range(1, len(results)) 
                 if results[i-1][4] == 1 and results[i][5] is not None]
    short_rets = [results[i][5] for i in range(1, len(results)) 
                  if results[i-1][4] == -1 and results[i][5] is not None]
    
    avg_long = np.mean(long_rets) if long_rets else 0
    avg_short = np.mean(short_rets) if short_rets else 0
    
    # --- Cumulative return (annualized) ---
    cum_ret = np.exp(np.sum(strat_rets)) - 1
    trading_days = len(strat_rets)
    years = trading_days / 252
    cum_ret_annualized = ((1 + cum_ret) ** (1 / years) - 1) if years > 0 else 0
    
    # --- Sharpe Ratio (annualized) ---
    excess_rets = strat_rets - rf_daily
    sharpe = np.sqrt(252) * np.mean(excess_rets) / np.std(excess_rets, ddof=1) if np.std(excess_rets) > 0 else 0
    
    # --- Sortino Ratio (annualized) ---
    downside_rets = excess_rets[excess_rets < 0]
    downside_std = np.std(downside_rets, ddof=1) if len(downside_rets) > 1 else 0
    sortino = np.sqrt(252) * np.mean(excess_rets) / downside_std if downside_std > 0 else 0
    
    # --- VaR at 5% ---
    var_5 = np.percentile(strat_rets, 5)
    
    # --- Print Results ---
    print("=" * 60)
    print("PERFORMANCE METRICS")
    print("=" * 60)
    print(f"Total trading days: {len(strat_rets)}")
    print(f"Total trades: {num_trades}")
    print(f"Trades per month: {trades_per_month:.2f}")
    print(f"")
    print(f"Mean daily return: {mean_ret:.6f} ({mean_ret*100:.4f}%)")
    print(f"Std daily return: {std_ret:.6f}")
    print(f"t-statistic: {t_stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    print(f"")
    print(f"Avg Long return: {avg_long:.6f} ({len(long_rets)} trades)")
    print(f"Avg Short return: {avg_short:.6f} ({len(short_rets)} trades)")
    print(f"")
    print(f"Cumulative return: {cum_ret:.4f} ({cum_ret*100:.2f}%)")
    print(f"Cumulative return (annualized): {cum_ret_annualized:.4f} ({cum_ret_annualized*100:.2f}%)")
    print(f"Sharpe ratio: {sharpe:.4f}")
    print(f"Sortino ratio: {sortino:.4f}")
    print(f"VaR (5%): {var_5:.6f} ({var_5*100:.4f}%)")
    print("=" * 60)
    
    return {
        'mean_return': mean_ret,
        'std_return': std_ret,
        't_stat': t_stat,
        'p_value': p_value,
        'trades_per_month': trades_per_month,
        'avg_long': avg_long,
        'avg_short': avg_short,
        'cumulative_return': cum_ret,
        'cumulative_return_annualized': cum_ret_annualized,
        'sharpe': sharpe,
        'sortino': sortino,
        'var_5': var_5
    }

## Cell 13: Jensen's Alpha (requires market returns)

In [100]:
def compute_jensens_alpha(strategy_returns, market_returns, rf_annual=0.05):
    """
    Compute Jensen's Alpha.
    Alpha = Rp - [Rf + Beta * (Rm - Rf)]
    """
    rf_daily = rf_annual / 252
    
    # Excess returns
    strat_excess = np.array(strategy_returns) - rf_daily
    mkt_excess = np.array(market_returns) - rf_daily
    
    # Beta = Cov(Rp, Rm) / Var(Rm)
    cov_matrix = np.cov(strat_excess, mkt_excess)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1]
    
    # Alpha
    alpha = np.mean(strat_excess) - beta * np.mean(mkt_excess)
    alpha_annual = alpha * 252
    
    print(f"Beta: {beta:.4f}")
    print(f"Jensen's Alpha (daily): {alpha:.6f}")
    print(f"Jensen's Alpha (annualized): {alpha_annual:.4f} ({alpha_annual*100:.2f}%)")
    
    return alpha, beta

In [99]:
def compare_to_benchmark(results, spy_test_data):
    """
    Compare strategy performance to buy-and-hold SPY benchmark.
    
    Args:
        results: Strategy results with [date, ticker, close, log_return, signal, strat_return]
        spy_test_data: Full SPY test data (same period as strategy test)
    
    This answers: "Did we beat buy-and-hold SPY over the test period?"
    """
    # Strategy returns (only where we had a position)
    strat_rets = [row[5] for row in results if row[5] is not None]
    strat_rets = np.array(strat_rets)
    
    # SPY buy-and-hold: sum ALL returns over the test period (not just trade dates)
    spy_rets = [row[6] for row in spy_test_data if row[6] is not None]
    spy_rets = np.array(spy_rets)
    
    # Cumulative returns (using log returns: sum then exp)
    cum_strategy = np.exp(np.sum(strat_rets)) - 1
    cum_benchmark = np.exp(np.sum(spy_rets)) - 1
    outperformance = cum_strategy - cum_benchmark
    
    # Annualize for comparison to 11% target
    strat_years = len(strat_rets) / 252 if len(strat_rets) > 0 else 1
    spy_years = len(spy_rets) / 252 if len(spy_rets) > 0 else 1
    cum_strategy_ann = ((1 + cum_strategy) ** (1/strat_years) - 1) if strat_years > 0 else 0
    cum_benchmark_ann = ((1 + cum_benchmark) ** (1/spy_years) - 1) if spy_years > 0 else 0
    
    print("="*60)
    print("BENCHMARK COMPARISON: Strategy vs. Buy-and-Hold SPY")
    print("="*60)
    print(f"Strategy cumulative return:  {cum_strategy*100:+.2f}% (annualized: {cum_strategy_ann*100:+.2f}%)")
    print(f"SPY cumulative return:       {cum_benchmark*100:+.2f}% (annualized: {cum_benchmark_ann*100:+.2f}%)")
    print(f"Outperformance:              {outperformance*100:+.2f}%")
    print(f"Target: >11% annualized")
    
    beats_spy = cum_strategy_ann > cum_benchmark_ann
    beats_target = cum_strategy_ann > 0.11
    
    if beats_spy and beats_target:
        print(f"\n[PASS] Strategy BEATS SPY and exceeds 11% target")
    elif beats_spy:
        print(f"\n[PARTIAL] Strategy beats SPY but below 11% target")
    elif beats_target:
        print(f"\n[PARTIAL] Strategy exceeds 11% but underperforms SPY")
    else:
        print(f"\n[FAIL] Strategy underperforms SPY and below 11% target")
    print("="*60)
    
    return {
        'cum_strategy': cum_strategy,
        'cum_strategy_annualized': cum_strategy_ann,
        'cum_benchmark': cum_benchmark,
        'cum_benchmark_annualized': cum_benchmark_ann,
        'outperformance': outperformance,
        'beats_spy': beats_spy,
        'beats_target': beats_target
    }

---
## PART 5: RUN THE STRATEGY
---

## Cell 14: Download and Prepare Data

In [5]:
# Download data for all tickers (3 stocks + 1 index)
all_data = {}
all_est_data = {}
all_test_data = {}

print("Downloading data for all symbols...")
import time

for symbol in ALL_SYMBOLS:
    print(f"  Fetching {symbol}...", end=" ")
    raw = get_daily_data(START_DATE, END_DATE, symbol, API_KEY)
    data = compute_log_returns(raw)
    est, test = partition_by_date(data, EST_START, EST_END)
    
    all_data[symbol] = data
    all_est_data[symbol] = est
    all_test_data[symbol] = test
    
    print(f"Est: {len(est)}, Test: {len(test)}")
    time.sleep(12)  # AlphaVantage rate limit (5 calls/min for free tier)

print("\n=== Data Summary ===")
for symbol in ALL_SYMBOLS:
    symbol_type = "INDEX" if symbol == INDEX else "STOCK"
    print(f"{symbol} ({symbol_type}): Est={len(all_est_data[symbol])}, Test={len(all_test_data[symbol])}")

  Fetching AAPL... Downloaded 1023 days of data for AAPL


NameError: name 'compute_log_returns' is not defined

## Cell 15: Generate Individual Signals

In [104]:
# Store Johnson-SU params and signals for each ticker
all_jsu_params = {}
all_signals = {}

for ticker in TICKERS:  # Only fit strategy on stocks, not index
    print(f"\n{'='*60}")
    print(f"Processing {ticker}")
    print(f"{'='*60}")
    
    est_data = all_est_data[ticker]
    test_data = all_test_data[ticker]
    
    # --- Fit Mean Reversion on estimation data ---
    est_returns = [row[6] for row in est_data if row[6] is not None]
    gamma, delta, xi, lam = fit_johnson_su(est_returns)
    all_jsu_params[ticker] = (gamma, delta, xi, lam)
    print(f"Johnson-SU params: gamma={gamma:.4f}, delta={delta:.4f}, xi={xi:.6f}, lam={lam:.6f}")
    
    # --- Generate signals on TEST data ---
    sig_mean_rev = signal_mean_reversion(test_data, gamma, delta, xi, lam, MEAN_REV_THRESHOLD)
    sig_rsi, rsi_values = signal_rsi(test_data, RSI_PERIOD, RSI_OVERSOLD, RSI_OVERBOUGHT)
    sig_atr, atr_values = signal_atr(test_data, ATR_PERIOD)
    sig_bb = signal_bollinger_bands(test_data)
    
    all_signals[ticker] = {
        'mean_rev': sig_mean_rev,
        'rsi': sig_rsi,
        'atr': sig_atr,
        'bb': sig_bb
    }
    
    # Count signals
    print(f"\nSignal counts on test data:")
    print(f"  Mean Reversion: {sum(1 for s in sig_mean_rev if s != 0)} signals")
    print(f"  RSI: {sum(1 for s in sig_rsi if s != 0)} signals")
    print(f"  ATR: {sum(1 for s in sig_atr if s != 0)} signals")
    print(f"  Bollinger: {sum(1 for s in sig_bb if s != 0)} signals")


Processing AAPL
Johnson-SU params: gamma=0.1296, delta=1.4834, xi=0.002559, lam=0.019909
Mean Reversion thresholds: lower=-0.027354, upper=0.026621

Signal counts on test data:
  Mean Reversion: 33 signals
  RSI: 69 signals
  ATR: 14 signals
  Bollinger: 49 signals

Processing MSFT
Johnson-SU params: gamma=0.1837, delta=1.6434, xi=0.003463, lam=0.023420
Mean Reversion thresholds: lower=-0.028317, upper=0.027139

Signal counts on test data:
  Mean Reversion: 15 signals
  RSI: 74 signals
  ATR: 7 signals
  Bollinger: 44 signals

Processing GOOGL
Johnson-SU params: gamma=0.1249, delta=1.4822, xi=0.002840, lam=0.023958
Mean Reversion thresholds: lower=-0.033064, upper=0.031947

Signal counts on test data:
  Mean Reversion: 20 signals
  RSI: 85 signals
  ATR: 8 signals
  Bollinger: 49 signals


## Cell 16: Combine Signals and Run Strategy

In [106]:
# Combine signals and compute returns for each ticker
# Methods available:
#   'core_confirm' - MR + (RSI or BB) confirmation (recommended)
#   'weighted'     - Weighted score >= 2.5
#   'consensus'    - 2 of 3 mean-rev indicators agree
#   'majority'     - Original: sign of sum of all 4

COMBINE_METHOD = 'majority'  # <- Change this to test different rules

all_results = {}
all_combined_signals = {}

for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"Strategy Execution: {ticker}")
    print(f"{'='*60}")
    
    sigs = all_signals[ticker]
    test_data = all_test_data[ticker]
    
    combined = combine_signals(
        [sigs['mean_rev'], sigs['rsi'], sigs['atr'], sigs['bb']],
        method=COMBINE_METHOD
    )
    all_combined_signals[ticker] = combined
    
    print(f"Combination method: {COMBINE_METHOD}")
    print(f"Combined signals: {sum(1 for s in combined if s != 0)} total")
    print(f"  Long signals: {sum(1 for s in combined if s == 1)}")
    print(f"  Short signals: {sum(1 for s in combined if s == -1)}")
    
    # Compute strategy returns
    results = compute_strategy_returns(test_data, combined)
    all_results[ticker] = results
    
    # Export to CSV with ticker in filename
    export_results(results, f'results_{ticker}.csv')
    print(f"Exported to results_{ticker}.csv")


Strategy Execution: AAPL
Combination method: majority
Combined signals: 113 total
  Long signals: 44
  Short signals: 69
Saved results to results_AAPL.csv
Exported to results_AAPL.csv

Strategy Execution: MSFT
Combination method: majority
Combined signals: 97 total
  Long signals: 42
  Short signals: 55
Saved results to results_MSFT.csv
Exported to results_MSFT.csv

Strategy Execution: GOOGL
Combination method: majority
Combined signals: 105 total
  Long signals: 26
  Short signals: 79
Saved results to results_GOOGL.csv
Exported to results_GOOGL.csv


## Cell 17: Evaluate Performance

In [107]:
# Get SPY returns for Jensen's alpha benchmark
spy_test_data = all_test_data[INDEX]
spy_dates = [row[0] for row in spy_test_data]
spy_returns = {row[0]: row[6] for row in spy_test_data}  # date -> log_return

all_metrics = {}

for ticker in TICKERS:
    print(f"\n{'#'*70}")
    print(f"# PERFORMANCE EVALUATION: {ticker}")
    print(f"{'#'*70}")
    
    results = all_results[ticker]
    
    # Compute performance metrics
    metrics = compute_performance_metrics(results)
    all_metrics[ticker] = metrics
    
    # Benchmark comparison: Do we beat buy-and-hold SPY?
    print("\n")
    benchmark = compare_to_benchmark(results, spy_test_data)
    
    # Jensen's Alpha: Risk-adjusted performance
    # Per assignment: "only use the market returns when a trade is closed"
    # A trade closes when the previous signal was non-zero (we held a position)
    trade_close_indices = []
    for i in range(1, len(results)):
        prev_signal = results[i-1][4]  # signal column
        if prev_signal != 0:  # Had a position that is now closing
            trade_close_indices.append(i)
    
    # Get SPY returns on trade-close dates
    strat_rets_filtered = []
    market_rets_filtered = []
    for i in trade_close_indices:
        date = results[i][0]
        strat_ret = results[i][5]
        if strat_ret is not None and date in spy_returns and spy_returns[date] is not None:
            strat_rets_filtered.append(strat_ret)
            market_rets_filtered.append(spy_returns[date])
    
    print(f"\n--- Jensen's Alpha (Trade-Close Dates Only, vs SPY) ---")
    print(f"Total trade-close days: {len(trade_close_indices)}")
    print(f"Days with valid SPY data: {len(market_rets_filtered)}")
    
    if len(market_rets_filtered) > 1:
        alpha, beta = compute_jensens_alpha(strat_rets_filtered, market_rets_filtered)
    else:
        print("Not enough trade-close dates for Jensen's Alpha calculation")

# Summary across all tickers
print(f"\n{'='*70}")
print("SUMMARY ACROSS ALL TICKERS")
print(f"{'='*70}")
print(f"{'Ticker':<10} {'Cum Ret':<12} {'Ann Ret':<12} {'Sharpe':<10} {'p-value':<10}")
print("-"*54)
for ticker in TICKERS:
    m = all_metrics[ticker]
    if m:
        print(f"{ticker:<10} {m['cumulative_return']*100:>8.2f}%   {m['cumulative_return_annualized']*100:>8.2f}%   {m['sharpe']:>8.4f}   {m['p_value']:>8.4f}")


######################################################################
# PERFORMANCE EVALUATION: AAPL
######################################################################
PERFORMANCE METRICS
Total trading days: 113
Total trades: 113
Trades per month: 8.69

Mean daily return: 0.000172 (0.0172%)
Std daily return: 0.024399
t-statistic: 0.0750
p-value: 0.9403

Avg Long return: 0.000846 (44 trades)
Avg Short return: -0.000257 (69 trades)

Cumulative return: 0.0197 (1.97%)
Cumulative return (annualized): 0.0444 (4.44%)
Sharpe ratio: -0.0170
Sortino ratio: -0.0275
VaR (5%): -0.035457 (-3.5457%)


BENCHMARK COMPARISON: Strategy vs. Buy-and-Hold SPY
Strategy cumulative return:  +1.97% (annualized: +4.44%)
SPY cumulative return:       +19.45% (annualized: +18.04%)
Outperformance:              -17.48%
Target: >11% annualized

[FAIL] Strategy underperforms SPY and below 11% target

--- Jensen's Alpha (Trade-Close Dates Only, vs SPY) ---
Total trade-close days: 113
Days with valid SPY data: 113


---
## NOTES / TODO
- [ ] Confirm ATR signal logic with professor
- [ ] Choose 4th indicator (currently Bollinger Bands)
- [ ] Tune indicator parameters
- [ ] Add visualizations
---